# Write-back — marquer les bénéficiaires servis

Quatrième et dernière des 4 étapes décrites dans [README.md](README.md), à exécuter
**une fois `../generate_new_codes.ipynb` passé** avec `SOURCE = 'FC'`.

`generate_new_codes.ipynb` recopie toutes les colonnes d'entrée et ne fait qu'en ajouter : le
CSV qu'il produit porte donc à la fois `eligibility_result_id` (venu de l'export SQL) et le
`id_psp` fraîchement fabriqué. Ce notebook en tire deux fichiers :

- `fc_2026_writeback.csv` — deux colonnes, entrée de `writeback_verdict.sql`, qui bascule les
  lignes concernées en verdict `eligible_pending_lca` ;
- `<...>-fc-prod.csv` — le CSV final **sans** la colonne technique, prêt pour l'injection en
  base de production.

C'est ce marquage qui rend tout le processus rejouable : `export_eligible_pending.sql` ne
ramasse que des `eligible_pending`, et `generate_new_codes.ipynb` ne déduplique que les
codes, jamais les personnes. Sans lui, un second passage fabriquerait un second code aux
mêmes bénéficiaires.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

notebook_dir = str(Path.cwd())
if notebook_dir not in sys.path:
    sys.path.append(notebook_dir)

import fc_pipeline as pipeline

load_dotenv()

# Le fichier daté écrit par ../generate_new_codes.ipynb, du type
# 'AAAA-MM-JJ-fc-with-codes.csv' et posé à côté de DB_FC_EXPORT_2026.
with_codes_filepath = os.environ['FC_WITH_CODES_PATHFILE_2026']

# Nom FIGÉ, attendu tel quel par writeback_verdict.sql et check_writeback.sql : \copy est la
# seule commande psql qui n'interpole aucune variable dans ses arguments, le chemin ne peut
# donc pas lui être passé. Il est écrit à côté des .sql, c'est-à-dire dans ce dossier.
writeback_filepath = str(Path.cwd() / pipeline.WRITEBACK_FILENAME)

prod_filepath = pipeline.prod_filepath_for(with_codes_filepath)

print(f"codes générés : {with_codes_filepath}")
print(f"write-back    : {writeback_filepath}")
print(f"prod          : {prod_filepath}")

In [ ]:
# Garde-fous puis découpage, dans fc_pipeline.split_writeback : un CSV sans
# eligibility_result_id ne vient pas de cette source, un id_psp vide écrirait un code vide en
# base. La cron (run_fc_pipeline.sh) appelle la même fonction, sous le nom
# `python fc_pipeline.py writeback`.
stats = pipeline.split_writeback(with_codes_filepath, writeback_filepath, prod_filepath)

pipeline.print_stats(stats)

## ⏭ Dernier geste, en base

Depuis **ce dossier**, tunnel Scalingo ouvert :

```bash
cd data/2026/partners/franceconnect
psql "$FC_DATABASE_URL" -f writeback_verdict.sql
psql "$FC_DATABASE_URL" -At -f check_writeback.sql   # doit afficher 0
```

`writeback_verdict.sql` affiche `lignes_csv`, `lignes_marquees` et `ids_introuvables` : les
deux premiers doivent être égaux. Rejoué, il met `UPDATE 0` — le filtre
`verdict = 'eligible_pending'` le rend idempotent.

`check_writeback.sql` compte, parmi les bénéficiaires de ce passage, ceux qui sont encore en
`eligible_pending` : toute valeur autre que 0 veut dire que des codes viennent d'être
fabriqués sans être marqués en base, et qu'un prochain passage en fabriquerait un second aux
mêmes personnes.

**Le CSV de production ne doit partir en injection qu'une fois ces deux contrôles passés** —
c'est exactement l'ordre que respecte la cron [run_fc_pipeline.sh](run_fc_pipeline.sh).